## Import libraries

In [1]:
import numpy as np
from itertools import combinations, product

## Input Evaluation Function

In [2]:
def evaluation(matrix):
    if not isinstance(matrix, (list, np.ndarray)):
        raise ValueError("Input must be a list or a NumPy ndarray.")

    try:
        matrix = np.asarray(matrix)
    except (TypeError, ValueError):
        raise ValueError("Input must be a rectangular 2D matrix.")

    if matrix.size == 0:
        raise ValueError("Matrix is empty.")

    if matrix.ndim != 2:
        raise ValueError("Matrix must be 2-dimensional.")

    if not (np.issubdtype(matrix.dtype, np.integer) or
            np.issubdtype(matrix.dtype, np.floating)):
        raise ValueError("Matrix entries must be real numeric values.")

    if not np.all(np.isin(matrix, [-1, 0, 1])):
        raise ValueError("A TU matrix can only contain -1, 0, and 1.")

    return matrix

## Generate Submatrices

In [3]:
def submatrix(matrix):
    rows_count, columns_count = matrix.shape
    max_size = min(rows_count, columns_count)

    for size in range(1, max_size + 1):
        row_combinations = combinations(range(rows_count), size)
        column_combinations = combinations(range(columns_count), size)

        for rows, columns in product(row_combinations, column_combinations):
            yield matrix[np.ix_(rows, columns)]

## Calculate the Determinant

In [4]:
def determinant_laplace(matrix):

    if hasattr(matrix, "tolist"):
        matrix = matrix.tolist()

    n = len(matrix)

    if n == 1:
        return matrix[0][0]

    if n == 2:
        return (
            matrix[0][0] * matrix[1][1]
            - matrix[0][1] * matrix[1][0]
        )

    determinant = 0

    for j in range(n):

        # Construct the minor matrix
        minor = [
            [matrix[i][k] for k in range(n) if k != j]
            for i in range(1, n)
        ]

        cofactor = (-1) ** j * matrix[0][j]

        determinant += cofactor * determinant_laplace(minor)

    return determinant


In [5]:
def determinant_bareiss(matrix):
    # Bareiss algorithm: exact determinant for integer matrices
    matrix = [[int(value) for value in row] for row in matrix.tolist()]

    n = len(matrix)

    if n == 1:
        return matrix[0][0]

    sign = 1
    previous_pivot = 1

    for k in range(n - 1):

        # If the pivot is zero, find a lower row to swap with it.
        if matrix[k][k] == 0:
            swap_row = None

            for row in range(k + 1, n):
                if matrix[row][k] != 0:
                    swap_row = row
                    break

            if swap_row is None:
                return 0

            matrix[k], matrix[swap_row] = matrix[swap_row], matrix[k]
            sign *= -1

        pivot = matrix[k][k]

        for i in range(k + 1, n):
            for j in range(k + 1, n):
                numerator = (
                    matrix[i][j] * pivot
                    - matrix[i][k] * matrix[k][j]
                )
                matrix[i][j] = numerator // previous_pivot

        for i in range(k + 1, n):
            matrix[i][k] = 0

        previous_pivot = pivot

    return sign * matrix[n - 1][n - 1]

## Executor Function

In [6]:
def is_totally_unimodular(matrix):
    evaluated = evaluation(matrix)

    for sub in submatrix(evaluated):
        determinant = determinant_bareiss(sub)

        if determinant not in (-1, 0, 1):
            return "This matrix is not Totally Unimodular"

    return "This matrix is Totally Unimodular" 

## Test

In [15]:
A1 = [
    [-1,0,0],
    [1,1,0],
    [0,-1,-1]
]

A2 = [
    [1,1,1,0,0,0,0],
    [0,1,1,1,0,0,0],
    [0,0,1,1,1,0,0],
    [0,0,0,1,1,1,0],
    [0,0,0,0,1,1,1]
]

A3 = [
    [1,1,0,1],    
    [1,2,0,1],    
    [0,0,0,1],    
    [0,0,1,0]    
]

A4 = [
    [1,0,1,-1],
    [1,1,0,-1],
    [0,-1,-1,0],
    [0,1,1,0],
]

In [22]:
try:
    print("A1:", is_totally_unimodular(A1))
except ValueError as e:
    print(e)

A1: This matrix is Totally Unimodular


In [23]:
try:
    print("A2:", is_totally_unimodular(A2))
except ValueError as e:
    print(e)

A2: This matrix is Totally Unimodular


In [24]:
try:
    print("A3:", is_totally_unimodular(A3))
except ValueError as e:
    print(e)

A TU matrix can only contain -1, 0, and 1.


In [27]:
try:
    print("A4:", is_totally_unimodular(A4))
except ValueError as e:
    print("A4:",e)

A4: This matrix is not Totally Unimodular


## Testing Area

In [32]:
M = [[0]]

try:
    print("Test_Matrix", is_totally_unimodular(M))
except ValueError as e:
    print("Test_Matrix", e)

Test_Matrix This matrix is Totally Unimodular


## Ask ChatGPT to test the algorithm

In [10]:
# A 1x1 TU matrix
assert is_totally_unimodular([[-1]]) == "This matrix is Totally Unimodular"

# Identity matrix is TU
assert is_totally_unimodular([[1, 0], [0, 1]]) == "This matrix is Totally Unimodular"

# This matrix has determinant -2, so it is not TU
assert determinant_laplace(np.array([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
])) == -2
assert is_totally_unimodular([
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
]) == "This matrix is not Totally Unimodular"

# Invalid values must be rejected
try:
    evaluation([[1, 2], [0, 1]])
except ValueError:
    pass
else:
    raise AssertionError("Invalid matrix was not rejected.")

# Empty and non-2D inputs must be rejected
for invalid_matrix in ([], [1, 0, -1]):
    try:
        evaluation(invalid_matrix)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid input was not rejected.")

print("All tests passed.")

All tests passed.
